# Module 1: Bayesian Decision Theory, MLE & Naive Bayes

**Z3003 Data Mining: Scaling to Large Data (BS Core) · IIT Madras Zanzibar**

Bernoulli/Gaussian MLE, Laplace-smoothed Categorical Naive Bayes, and log-probability classification.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inyalala/iitmz-teaching-site/blob/main/courses/dm-scale/labs/DM01_Bayesian_Classification_MLE.ipynb)

Run cells top to bottom (`Runtime > Run all`). Every method is self-contained and verified with assertions.

## 1. Maximum Likelihood Estimation (MLE) & Bayesian Risk

For Bernoulli trials, $\hat{p}_{\text{MLE}} = \frac{1}{n}\sum x_i$. Bayes' theorem updates prior probabilities via $P(C\mid x) \propto P(C)P(x\mid C)$.

In [1]:
import math
import numpy as np

rng = np.random.default_rng(5)
bernoulli_samples = rng.binomial(n=1, p=0.8, size=500)
p_mle = float(np.mean(bernoulli_samples))
print(f"Bernoulli MLE (n=500, true p=0.80): p_hat = {p_mle:.4f}")
assert abs(p_mle - 0.80) < 0.05

# Malaria screening example (Week 6 Summary):
# prevalence=0.02, sensitivity P(+|sick)=0.95, false positive P(+|healthy)=0.05
p_sick = 0.02
p_pos_given_sick = 0.95
p_pos_given_healthy = 0.05
p_pos = p_pos_given_sick * p_sick + p_pos_given_healthy * (1 - p_sick)
p_sick_given_pos = (p_pos_given_sick * p_sick) / p_pos
print(f"Bayes malaria check: P(sick | positive) = {p_sick_given_pos:.3f}")
assert abs(p_sick_given_pos - 190 / 680) < 1e-9


Bernoulli MLE (n=500, true p=0.80): p_hat = 0.7980
Bayes malaria check: P(sick | positive) = 0.279


## 2. Categorical Naive Bayes with Laplace Smoothing & Gaussian Naive Bayes

Laplace smoothing `(count + 1) / (class_size + k)` eliminates zero-count multiplication collapse, while log-probabilities prevent numerical underflow.

In [2]:
from collections import Counter, defaultdict


class CategoricalNaiveBayes:
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y):
        n = len(y)
        self.counts = Counter(y)
        self.classes = sorted(self.counts)
        self.cats = defaultdict(set)
        vc = defaultdict(Counter)
        for row, c in zip(X, y):
            for j, v in enumerate(row):
                self.cats[j].add(v)
                vc[(c, j)][v] += 1
        self.log_prior = {c: math.log(self.counts[c] / n) for c in self.classes}
        self.log_lik = {}
        for c in self.classes:
            for j in range(len(X[0])):
                k = len(self.cats[j])
                denom = self.counts[c] + self.alpha * k
                for v in self.cats[j]:
                    self.log_lik[(c, j, v)] = math.log((vc[(c, j)][v] + self.alpha) / denom)
        return self

    def predict_proba(self, row):
        ls = {c: self.log_prior[c] + sum(self.log_lik[(c, j, v)] for j, v in enumerate(row)) for c in self.classes}
        m = max(ls.values())
        ex = {c: math.exp(v - m) for c, v in ls.items()}
        s = sum(ex.values())
        return {c: v / s for c, v in ex.items()}


X_sms = [["free", "win"], ["free", "offer"], ["free", "click"], ["win", "cash"], ["hello", "tea"], ["class", "notes"], ["farm", "rain"], ["tea", "later"]]
y_sms = ["spam", "spam", "spam", "spam", "ham", "ham", "ham", "ham"]
cnb = CategoricalNaiveBayes(alpha=1.0).fit(X_sms, y_sms)
p = cnb.predict_proba(["free", "win"])
print("Posterior for ['free', 'win']:", {k: round(v, 3) for k, v in p.items()})
assert p["spam"] > 0.75


Posterior for ['free', 'win']: {'ham': 0.111, 'spam': 0.889}
